In [31]:
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import matplotlib.pyplot as plt
import seaborn as sns

train = pd.read_csv('/kaggle/input/datasets/aminelayed/barhoum/train.csv')
test = pd.read_csv('/kaggle/input/datasets/aminelayed/barhoum/test.csv')

train.head()
train.info()
train.describe()
train.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 891 entries, 0 to 890
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  891 non-null    int64  
 1   Survived     891 non-null    int64  
 2   Pclass       891 non-null    int64  
 3   Name         891 non-null    object 
 4   Sex          891 non-null    object 
 5   Age          714 non-null    float64
 6   SibSp        891 non-null    int64  
 7   Parch        891 non-null    int64  
 8   Ticket       891 non-null    object 
 9   Fare         891 non-null    float64
 10  Cabin        204 non-null    object 
 11  Embarked     889 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 83.7+ KB


PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age            177
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64

In [32]:
# Extract title from Name
train['Title'] = train['Name'].str.extract(r' ([A-Za-z]+)\.', expand=False)
train['Title'] = train['Title'].replace(['Lady','Countess','Capt','Col','Don',
                                          'Dr','Major','Rev','Sir','Jonkheer','Dona'], 'Rare')
train['Title'] = train['Title'].replace('Mlle', 'Miss')
train['Title'] = train['Title'].replace('Ms', 'Miss')
train['Title'] = train['Title'].replace('Mme', 'Mrs')

# Family size
train['FamilySize'] = train['SibSp'] + train['Parch'] + 1
train['IsAlone'] = (train['FamilySize'] == 1).astype(int)

# Fill missing values
train['Age'] = train.groupby('Title')['Age'].transform(
    lambda x: x.fillna(x.median()))
train['Embarked'] = train['Embarked'].fillna(train['Embarked'].mode()[0])

# Drop useless columns
train = train.drop(['PassengerId','Name','Ticket','Cabin'], axis=1)

# Encode categoricals
train = pd.get_dummies(train, columns=['Sex','Embarked','Title'])

print(train.head())
print(train.shape)


   Survived  Pclass   Age  SibSp  Parch     Fare  FamilySize  IsAlone  \
0         0       3  22.0      1      0   7.2500           2        0   
1         1       1  38.0      1      0  71.2833           2        0   
2         1       3  26.0      0      0   7.9250           1        1   
3         1       1  35.0      1      0  53.1000           2        0   
4         0       3  35.0      0      0   8.0500           1        1   

   Sex_female  Sex_male  Embarked_C  Embarked_Q  Embarked_S  Title_Master  \
0       False      True       False       False        True         False   
1        True     False        True       False       False         False   
2        True     False       False       False        True         False   
3        True     False       False       False        True         False   
4       False      True       False       False        True         False   

   Title_Miss  Title_Mr  Title_Mrs  Title_Rare  
0       False      True      False       False  


In [33]:
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

X = train.drop('Survived', axis=1)
Y = train['Survived']

X_train, X_val, Y_train, Y_val = train_test_split(X, Y, test_size=0.2, random_state=42)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, Y_train)

# Validate
val_predictions = model.predict(X_val)
print("Validation Accuracy:", accuracy_score(Y_val, val_predictions))

# Cross validation — more reliable than single split
cv_scores = cross_val_score(model, X, Y, cv=5)
print("CV Mean:", cv_scores.mean(), "Std:", cv_scores.std())

Validation Accuracy: 0.8435754189944135
CV Mean: 0.7991212102190698 Std: 0.026283918207960673


NameError: name 'passenger_ids' is not defined